# Lab Instructions

In the lab, you're presented a task such as building a dataset, training a model, or writing a training loop, and we'll provide the code structured in such a way that you can fill in the blanks in the code using the knowledge you acquired in the chapters that precede the lab. You should be able to find appropriate snippets of code in the course content that work well in the lab with minor or no adjustments.

The blanks in the code are indicated by ellipsis (`...`) and comments (`# write your code here`).

In some cases, we'll provide you partial code to ensure the right variables are populated and any code that follows it runs accordingly.

```python
# write your code here
x = ...
```

The solution should be a single statement that replaces the ellipsis, such as:

```python
# write your code here
x = [0, 1, 2]
```

In some other cases, when there is no new variable being created, the blanks are shown like in the example below:

```python
# write your code here
...
```

Although we're showing you only a single ellipsis (`...`), you may have to write more than one line of code to complete the step, such as:

```python
# write your code here
for i, xi in enumerate(x):
    x[i] = xi * 2
```

## Installation Notes

To run this notebook on Google Colab, you will need to install the following library: evaluate.

In Google Colab, you can run the following command to install this library:

In [ ]:
!pip install evaluate

## 8.5 Lab 4: Sentiment Analysis

In this lab, you'll fine-tune an encoder-based model to perform sentiment analysis on the Standford Sentiment Treebank (SST2) dataset. You'll load RoBERTa's sibling, DistilRoBERTa, use its prescribed transformations to preprocess text in the SST2 dataset, and fine-tune (train) it for one epoch.

### 8.5.1 Model

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/model_step1.png)

You'll use Hugging Face's `AutoModelForSequenceClassification` to perform binary classification (we have two classes, "positive" and "negative" sentiment).

In [ ]:
from transformers import AutoModelForSequenceClassification
repo_id = "distilbert/distilroberta-base"

id2label = {0: "negative", 1: "positive"}
label2id = {label: idx for idx, label in id2label.items()}

# write your code here
model = ...
model

Notice that the model has two main parts: roberta and classifier. The former is the original base model, and the latter is the newly created head which we’ll train shortly. You can inspect them by simply using dot notation:

### 8.5.2 Dataset

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/data_step1.png)

Now, you will load Hugging Face's ["Stanford Sentiment Treebank (SST2)"](https://huggingface.co/datasets/stanfordnlp/sst2) dataset. It is already split into train, validation, and test sets. Make sure to shuffle the training set first, and then select only 12,000 samples from it to keep training manageable.

In [ ]:
from datasets import load_dataset, DatasetDict

# write your code here
raw_datasets = ...
train_ds = ...
val_ds = ...

datasets = DatasetDict({"train": train_ds, "val": val_ds})
datasets

Let's take a look at one data point from the SST2 dataset. Just run the code below as is to visualize the output:

In [ ]:
row = datasets['train'][0]
text, label = row['sentence'], row['label']
text, label

Each data point is a dictionary, containing a line of text, and the corresponding label - the sentiment (0 for negative, 1 for positive).

### 8.5.3 Tokenizer

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/data_step3.png)

You already know the drill: you must preprocess the input (the text) using the prescribed transformation for the model you're using, so it gets tokenized, converted into token ids, and prependend/appended with the appropriate special tokens.

Load DistilRoBERTa's tokenizer and write a function that takes a dictionary with the `sentence` key (it may have other keys as well) and returns a dictionary with `input_ids`, `attention_mask` keys (remember that the `map()` method of HF datasets work by _merging_ dictionaries):

In [ ]:
from transformers import AutoTokenizer

# write your code here
tokenizer = ...

In [ ]:
def tokenize_batch(batch):
    # write your code here
    return ...

Now, apply the function to every batch in our datasets (set `batched=True` when calling `map`):

In [ ]:
# write your code here
tokenized = ...
tokenized

To keep our datasets tidy, selct only the columns we're interested in (`input_ids` and `label`):

In [ ]:
# write your code here
tokenized = ...
tokenized['train'][:3]

Did you notice the transformation is returning a regular Python list of token ids, not a PyTorch tensor? Remember, we cannot make a tensor out of lists of different lengths. The solution? Padding the shorter sentences, so they all have the same length.

But, how can we think of padding sentences if we don't have a mini-batch yet? We delegate this job to the dataloaders's collate function!

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/data_step5.png)

So far, we've been using data loaders without specifying a collate function, that is, we're using its default collate function. For tabular data, the default collator is more than enough. It simply stacks several data points together and, since they all have the same size, it works smoothly. But this strategy breaks apart when we're dealing with sequences of different lengths, as we've already experienced while trying to make a tensor out of them.

Just like before, padding is the solution for our problem, and we're using a [collator](https://huggingface.co/learn/nlp-course/en/chapter3/2#dynamic-padding) designed to automatically pad the sequences before stacking them together: HF's `DataCollatorWithPadding`. It takes the tokenizer as an argument in order to determine which token is the padding token, and which side (left or right) should be padded.

Let's try it on a slice of four sequences from our training set:

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
data_collator(tokenized['train'][:4])

You can easily recognize the padding tokens sitting at the right end of the sequences (a sequence of ones). Moreover, the location of every padding token is indicated by the sequence's corresponding attention mask. The masks tell the model which tokens should be considered (value of one) or ignored (value of zero).

Next, let's assign this data collator to each dataloader:

In [ ]:
from torch.utils.data import DataLoader

dataloaders = {}
# write your code here
dataloaders['train'] = DataLoader(dataset=tokenized['train'], batch_size=32, shuffle=True, collate_fn=data_collator)
dataloaders['val'] = DataLoader(dataset=tokenized['val'], batch_size=64, collate_fn=data_collator)

Now, let's fetch a mini-batch from our data loader (just run the code below as is to visualize the output):

In [ ]:
batch = next(iter(dataloaders["train"]))
{key: value.shape for key, value in batch.items()}

As you can see, there are plenty of padding tokens there. The collator will always pad the sequences to match the longest sequence in a particular mini-batch. This means that mini-batches may have sequences of different lengths (when compared across mini-batches but not inside the same one).

### 8.5.4 Training the Classifier Head Only

Now, it is time to write a training loop to fine-tune your DistilRoBERTa model on the SST2 dataset. This is a large model, but we’re only training its head (`model.classifier`), and the training set has only 12,000 data points, so you can train it over three epochs.

Since the base model (`model.roberta`) is not being trained at this point, we can freeze its parameters by turning off gradient computation for them:

In [ ]:
for parameter in model.roberta.parameters():
    parameter.requires_grad = False

#### 8.5.4.1 Loss Function

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/model_step2.png)

Sentiment analysis is a classification task, so we need to use the appropriate loss function for the task. Even though it is a binary classification, RoBERTa's classification head is actually producing two logits instead of one, so you have to use `CrossEntropyLoss` (which can handle two or more logits).

In [ ]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()

This step is actually redundant now. Since we're using a HF model, the loss is automatically returned when the model is in training model. We simply retrieve the loss from the output's `loss` attribute.

#### 8.5.4.2 Optimizer

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/model_step3.png)

Although `Adam` is the optimizer of choice, we suggest you try out `AdamW`, a modified version that is also commonly used. At this point, we’re only interested in training the head (`model.classifier`), so make sure the optimizer handles these parameters only.

In [ ]:
import torch.optim as optim

# suggested learning rate
lr = 1e-3
# write your code here
optimizer = ...

#### 8.4.4.2 Training Loop

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/model_step4.png)

Now, it's your turn to write the missing parts of the training loop below.

In [ ]:
import torch
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model.to(device)

num_epochs = 3

for epoch in range(num_epochs):
    # Set the model's mode
    # write your code here
    ...
    # the base model must behave deterministically
    # so we set it to evaluation mode, as only the
    # head itself is being trained
    model.roberta.eval()
    running_loss = 0.0

    for batch in tqdm(dataloaders["train"], leave=False):
        # Send batch features and targets to the device
        # write your code here
        batch = ...

        # Step 1 - forward pass
        # write your code here
        output = ...

        # Step 2 - computing the loss
        # write your code here
        loss = ...

        # Step 3 - computing the gradients
        # Tip: it requires a single method call to backpropagate gradients
        # write your code here
        ...

        # Step 4 - updating parameters and zeroing gradients
        # Tip: it takes two calls to optimizer's methods
        # write your code here
        ...

        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloaders["train"])
    print(f"Epoch {epoch + 1:02d} | loss: {epoch_loss:.4f}")

### 8.5.5 Evaluation

Write a function that takes a model, a dataloader, and a list of metrics, and proceeds to compute the metrics using HF’s evaluate package. Then, call the function to evaluate the fine-tuned model.

In [ ]:
def evaluate_metrics(model, dataloader, metrics): 
    device = next(iter(model.parameters())).device      
    for batch in dataloader:
        # write your code here
        labels = ...
        # send batch to the device
        # write your code here
        batch = {key: value.to(device) for key, value in batch.items()}

        model.eval()
        with torch.inference_mode():
            # write your code here
            predictions = ...

        # add a batch of predictions and labels to each metric
        # write your code here
        ...

    # We specify the average=None argument during compute for precision, recall, and f1
    result = []
    for metric in metrics:
        if metric.name in ["precision", "recall", "f1"]:
            result.append(metric.compute(average=None))
        else:
            result.append(metric.compute())
    return result

In [ ]:
stage1_metrics = evaluate_metrics(model, dataloaders["val"], [metric1, metric2, metric3, metric4])
stage1_metrics

### 8.5.6 Training the Head and the Last Transformer Block

So far, we haven’t touched the original base model, we’ve only used it as a feature extractor to train the classifier head. However, we can make our fine-tuned model more accurate if we unfreeze parts of the base model as well, so more parameters can be effectively fine-tuned. The prime candidate is the very last Transformer block, the one that sits right before the classifier head:

In [ ]:
model.roberta.encoder.layer[-1]

If we’re comparing the two approaches, we must reload the original model, so we can start training using a randomly initialized classifier head once again.
Write code to reload the model, freeze the original base model (model.roberta), and then unfreeze the last Transformer block only.

In [ ]:
# write your code here
model = ...

In [ ]:
# freeze the original base model
# write your code here
...

# unfreeze the last Transformer block
# write your code here
...

#### 8.5.6.1 Optimizer

For this second training run, we’re training more than the classifier head, so the optimizer must account for every parameter being updated, both in the head and in the last Transformer block.

We’re keeping the same learning rate as before when it comes to the classifier head, which is randomly initialized. The remaining, however, are already pretrained, so we’re better off using a much lower learning rate for them. Luckily, we can pick and choose learning rates to different subsets of parameters, as shown below:

In [ ]:
optimizer = optim.AdamW([
  {"params": model.classifier.parameters(), "lr": 1e-3},
  {"params": model.roberta.encoder.layer[-1].parameters(), "lr": 5e-5}
])

#### 8.5.6.2 Training Loop

It is time to write another training loop. Although you may be tempted to copy and paste the former training loop, this one has one extra line of code that you must keep to make it work properly. Can you identify the line? Once you do, you can simply add it back to the copied and pasted code, as everything else is exactly the same.

In [ ]:
model.to(device)

num_epochs = 3

for epoch in range(num_epochs):
    # Set the model's mode
    # write your code here
    ...
    # the base model must behave deterministically
    # so we set it to evaluation mode
    model.roberta.eval()
    # but we unfroze the last block
    model.roberta.encoder.layer[-1].train()
    running_loss = 0.0

    for batch in tqdm(dataloaders["train"], leave=False):
        # Send batch features and targets to the device
        # write your code here
        batch = ...

        # Step 1 - forward pass
        # write your code here
        output = ...

        # Step 2 - computing the loss
        # write your code here
        loss = ...

        # Step 3 - computing the gradients
        # Tip: it requires a single method call to backpropagate gradients
        # write your code here
        ...

        # Step 4 - updating parameters and zeroing gradients
        # Tip: it takes two calls to optimizer's methods
        # write your code here
        ...

        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloaders["train"])
    print(f"Epoch {epoch + 1:02d} | loss: {epoch_loss:.4f}")

#### 8.5.6.3 Evaluation and Comparison

How much better is the newly fine-tuned model? Call the evaluation function once again, and run the comparison code.

In [ ]:
# write your code here
stage2_metrics = ...
stage2_metrics

In [ ]:
import pandas as pd

results = pd.DataFrame([
  {"strategy": "Classifier head only",
   "trainable_parameters": stage1_trainable/1e6,
   "trainable_fraction": 100*stage1_trainable / stage1_total,
   "validation_accuracy": 100*stage1_metrics[0]['accuracy']},
  {"strategy": "Last block + classifier",
   "trainable_parameters": stage2_trainable/1e6,
   "trainable_fraction": 100*stage2_trainable / stage2_total,
   "validation_accuracy": 100*stage2_metrics[0]['accuracy']}
])
results

### 8.5.8 Inference

![](https://raw.githubusercontent.com/dvgodoy/assets/main/PyTorchInPractice/images/ch0/model_step5.png)

Write a function that takes a list of sentences, a model, and its tokenizer, and returns the most likely category and the corresponding probability. Since you're handling several sequences, do not forget to pad and truncate them (to a maximum of 128 tokens each).

The model returns two logits, one for each class, so you must use the softmax function to convert them into probabilities.

In [ ]:
def predict(texts, model, tokenizer):
    # Build a tensor of token ids out of the input sequence
    # write your code here
    inputs = ...

    # Set the model to the appropriate mode
    # write your code here
    ...

    device = next(iter(model.parameters())).device
    # Tip: Don't forget to send the inputs to the same device as the model
    # write your code here
    inputs = ...

    # Use the model to make predictions/logits
    # write your code here
    with torch.inference_mode():
        logits = ...

    # Compute the probabilities corresponding to the logits
    # and return the top value and index
    # write your code here
    probabilities = ...
    confidences, predictions = ...
    
    return [{"label": id2label[idx.item()], "confidence": conf.item()}
            for idx, conf in zip(predictions, confidences)]

Now, try out your prediction function and fine-tuned model (just run the code cells below as they are to visualize their outputs):

In [ ]:
categories = ['negative', 'positive']
texts = ["I am really liking this course", "This course is too complicated!"]
predict(texts, model, tokenizer)

That's cool, but what if we could perform sentiment analysis out-of-the-box? That's what we'll do in the second part of Pretrained Models for Natural Language Processing.